In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

CHUNKS_PATH = DATA_DIR / "rag_chunks.parquet"
META_OUT_PATH = DATA_DIR / "rag_chunks_metadata.parquet"
EMB_OUT_PATH = DATA_DIR / "rag_chunk_embeddings.npy"

CHUNKS_PATH, META_OUT_PATH, EMB_OUT_PATH


(PosixPath('data/rag_chunks.parquet'),
 PosixPath('data/rag_chunks_metadata.parquet'),
 PosixPath('data/rag_chunk_embeddings.npy'))

In [2]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
print(chunks_df.shape)
chunks_df.head()


(2173, 6)


,chunk_id,article_id,source,title,chunk_index,chunk_text
0,1_0,1,Al Jazeera,What is the political agenda of artificial int...,0,Could AI single-handedly decide the course of ...
1,1_1,1,Al Jazeera,What is the political agenda of artificial int...,1,"of hyperrealistic, AI-generated content, such ..."
2,1_2,1,Al Jazeera,What is the political agenda of artificial int...,2,all era-defining technology – comes with consi...
3,1_3,1,Al Jazeera,What is the political agenda of artificial int...,3,"like 2001: A Space Odyssey, Blade Runner and T..."
4,1_4,1,Al Jazeera,What is the political agenda of artificial int...,4,a clear distinction between symbolic machine ‘...


In [4]:
from sentence_transformers import SentenceTransformer

# Strong, general-purpose retrieval model
embed_model_name = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(embed_model_name)

embedder


/Users/shoaib/Desktop/University/5th Semester/NLP/Project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-30 20:21:39.211662: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [5]:
texts = chunks_df["chunk_text"].astype(str).tolist()
len(texts)


2173

In [6]:
# encode in batches to avoid RAM spikes
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # cosine similarity friendly
)

embeddings = embeddings.astype("float32")  # smaller on disk

print("Embeddings shape:", embeddings.shape)


Batches: 100%|██████████| 68/68 [11:14<00:00,  9.92s/it]

Embeddings shape: (2173, 768)


In [7]:
# Save metadata (everything except the big embeddings)
meta_cols = ["chunk_id", "article_id", "source", "title", "chunk_index", "chunk_text"]
meta_df = chunks_df[meta_cols].copy()

meta_df.to_parquet(META_OUT_PATH, index=False)
np.save(EMB_OUT_PATH, embeddings)

print("Saved metadata to:", META_OUT_PATH)
print("Saved embeddings to:", EMB_OUT_PATH)
print("Meta shape:", meta_df.shape)


Saved metadata to: data/rag_chunks_metadata.parquet
Saved embeddings to: data/rag_chunk_embeddings.npy
Meta shape: (2173, 6)
